In [1]:
pip install SoccerNet --upgrade

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install ultralytics opencv-python


Note: you may need to restart the kernel to use updated packages.


In [3]:
from SoccerNet.utils import getListGames

# Cargar todos los juegos disponibles
all_games = getListGames(split=["train", "valid", "test", "challenge"])

# Filtrar por liga y temporada
def filter_games(games, league_keyword, seasons=[]):
    filtered = [g for g in games if league_keyword in g]
    if seasons:
        filtered = [g for g in filtered if any(season in g for season in seasons)]
    return filtered

# Buscar por liga y año
epl_games = filter_games(all_games, "england_epl", ["2018", "2017", "2016"])
laliga_games = filter_games(all_games, "spain_laliga", ["2018", "2017", "2016"])

print(f"Partidos EPL recientes: {len(epl_games)}")
print(epl_games[:10])  # Ver algunos ejemplos

print(f"Partidos LaLiga recientes: {len(laliga_games)}")
print(laliga_games[:10])


Partidos EPL recientes: 98
['england_epl\\2015-2016\\2015-08-08 - 19-30 Chelsea 2 - 2 Swansea', 'england_epl\\2015-2016\\2015-08-29 - 17-00 Chelsea 1 - 2 Crystal Palace', 'england_epl\\2015-2016\\2015-08-29 - 17-00 Manchester City 2 - 0 Watford', 'england_epl\\2015-2016\\2015-09-12 - 14-45 Everton 3 - 1 Chelsea', 'england_epl\\2015-2016\\2015-09-12 - 17-00 Crystal Palace 0 - 1 Manchester City', 'england_epl\\2015-2016\\2015-09-19 - 19-30 Manchester City 1 - 2 West Ham', 'england_epl\\2015-2016\\2015-09-26 - 17-00 Liverpool 3 - 2 Aston Villa', 'england_epl\\2015-2016\\2015-10-17 - 17-00 Chelsea 2 - 0 Aston Villa', 'england_epl\\2015-2016\\2015-10-31 - 15-45 Chelsea 1 - 3 Liverpool', 'england_epl\\2015-2016\\2015-11-07 - 18-00 Manchester United 2 - 0 West Brom']
Partidos LaLiga recientes: 99
['spain_laliga\\2015-2016\\2015-08-29 - 21-30 Barcelona 1 - 0 Malaga', 'spain_laliga\\2015-2016\\2015-09-12 - 17-00 Espanyol 0 - 6 Real Madrid', 'spain_laliga\\2015-2016\\2015-09-23 - 22-00 Ath Bilba

In [ ]:
from SoccerNet.Downloader import SoccerNetDownloader
mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory="../data/tracking/SoccerNet")
mySoccerNetDownloader.downloadDataTask(task="tracking", split=["train","test","challenge"])
mySoccerNetDownloader.downloadDataTask(task="tracking-2023", split=["train", "test", "challenge"])

C:\Users\alejo\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

In [28]:
import os
import configparser
from pathlib import Path
import pandas as pd

# === Ruta de la secuencia a probar ===


sequence_dir = Path("../data/tracking/SoccerNet/tracking/test/test/SNMOT-116").resolve()

gt_path = os.path.join(sequence_dir, "gt", "gt.txt")
gameinfo_path = os.path.join(sequence_dir, "gameinfo.ini")
seqinfo_path = os.path.join(sequence_dir, "seqinfo.ini")
output_dir = os.path.join(sequence_dir, "labels")
os.makedirs(output_dir, exist_ok=True)

# === Leer resolución del video desde seqinfo.ini ===
config = configparser.ConfigParser()
config.read(seqinfo_path)

try:
    img_width = int(config["Sequence"]["imWidth"])
    img_height = int(config["Sequence"]["imHeight"])
except KeyError as e:
    raise ValueError(f"No se pudo leer imWidth o imHeight: {e}")

# === Leer mapeo de tracklets (jugadores, balón, etc.) desde gameinfo.ini ===
tracklet_class_map = {}
tracklet_team_map = {}

class_name_to_id = {
    "player team left": 0,
    "player team right": 1,
    "goalkeeper team left": 2,
    "goalkeeper team right": 3,
    "referee": 4,
    "ball": 5
}

with open(gameinfo_path, "r") as f:
    for line in f:
        if line.startswith("trackletID_"):
            parts = line.strip().split("=")
            track_id = int(parts[0].split("_")[1])
            class_info = parts[1].split(";")[0].strip()
            class_id = class_name_to_id.get(class_info, -1)
            if class_id != -1:
                tracklet_class_map[track_id] = class_id
                # también guardamos si es team left / right
                if "left" in class_info:
                    tracklet_team_map[track_id] = "left"
                elif "right" in class_info:
                    tracklet_team_map[track_id] = "right"
                else:
                    tracklet_team_map[track_id] = "other"

# === Procesar gt.txt y convertir anotaciones por frame ===
frame_annotations = {}

with open(gt_path, "r") as f:
    for line in f:
        parts = line.strip().split(",")
        if len(parts) < 6:
            continue
        frame_id, track_id, x, y, w, h = map(float, parts[:6])
        frame_id = int(frame_id)
        track_id = int(track_id)

        if track_id not in tracklet_class_map:
            continue

        class_id = tracklet_class_map[track_id]
        team_label = tracklet_team_map[track_id]

        # Coordenadas normalizadas formato YOLO
        x_center = (x + w / 2) / img_width
        y_center = (y + h / 2) / img_height
        w_norm = w / img_width
        h_norm = h / img_height

        # Incluir class_id y track_id para tracking individual
        yolo_line = f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f} {track_id} # team:{team_label}"
        frame_file = os.path.join(output_dir, f"{frame_id:06}.txt")
        frame_annotations.setdefault(frame_file, []).append(yolo_line)

# === Guardar anotaciones por frame ===
for path, lines in frame_annotations.items():
    with open(path, "w") as f:
        f.write("\n".join(lines))


f"Proceso completado para {len(frame_annotations)} frames."



'Proceso completado para 750 frames.'

In [29]:
import os
import cv2
from pathlib import Path

# === RUTA BASE ===
sequence_dir = Path("../data/tracking/SoccerNet/tracking/test/test/SNMOT-116").resolve()
frames_dir = sequence_dir / "img1"          # carpeta con frames (jpg o png)
labels_dir = sequence_dir / "labels"        # carpeta con txt generados en formato YOLO
output_dir = sequence_dir / "frames_bbox"   # carpeta donde guardar imágenes con bounding boxes
os.makedirs(output_dir, exist_ok=True)

# === COLORES POR CLASE (ajusta si quieres) ===
colors = {
    0: (0, 255, 0),      # player left - verde
    1: (0, 0, 255),      # player right - rojo
    2: (255, 255, 0),    # GK left - cyan
    3: (255, 0, 255),    # GK right - magenta
    4: (255, 165, 0),    # referee - naranja
    5: (255, 255, 255)   # ball - blanco
}

# === PROCESAR TODOS LOS FRAMES ===
for label_file in sorted(labels_dir.glob("*.txt")):
    frame_id = label_file.stem
    frame_path = frames_dir / f"{frame_id}.jpg"
    if not frame_path.exists():
        continue

    img = cv2.imread(str(frame_path))
    if img is None:
        continue
    h, w = img.shape[:2]

    with open(label_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6:
                continue
            class_id, x_center, y_center, box_w, box_h, track_id = map(float, parts[:6])
            class_id = int(class_id)
            track_id = int(track_id)

            # Bounding box
            x1 = int((x_center - box_w / 2) * w)
            y1 = int((y_center - box_h / 2) * h)
            x2 = int((x_center + box_w / 2) * w)
            y2 = int((y_center + box_h / 2) * h)

            color = colors.get(class_id, (100, 100, 100))
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, f"ID:{track_id}", (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            cv2.putText(img, f"{class_id}", (x1, y2 + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    # Guardar resultado
    cv2.imwrite(str(output_dir / f"{frame_id}.jpg"), img)

print("✅ Visualización de bounding boxes completa.")


✅ Visualización de bounding boxes completa.


In [ ]:
import os

# Límite de tamaño de GitHub por archivo (100 MB)
LIMITE_MB = 100
LIMITE_BYTES = LIMITE_MB * 1024 * 1024

def revisar_archivos_en_main(ruta_repositorio):
    ruta_main = os.path.join(ruta_repositorio, '../')
    
    if not os.path.exists(ruta_main):
        print(f"❌ No se encontró la carpeta 'main' en: {ruta_repositorio}")
        return

    print(f"🔍 Revisando archivos en: {ruta_main}\n")
    archivos_grandes = []

    for carpeta_actual, _, archivos in os.walk(ruta_main):
        for archivo in archivos:
            ruta_archivo = os.path.join(carpeta_actual, archivo)
            try:
                tamaño = os.path.getsize(ruta_archivo)
                if tamaño > LIMITE_BYTES:
                    archivos_grandes.append((ruta_archivo, tamaño))
            except Exception as e:
                print(f"⚠️ Error al acceder a {ruta_archivo}: {e}")

    if archivos_grandes:
        print("⚠️ Archivos que superan el límite de 100 MB:\n")
        for ruta, tamaño in archivos_grandes:
            print(f"📁 {ruta} — {tamaño / (1024 * 1024):.2f} MB")
    else:
        print("✅ Todos los archivos están dentro del límite permitido.")

if __name__ == "__main__":
    ruta = input("📂 Introduce la ruta del repositorio local: ").strip()
    revisar_archivos_en_main(ruta)
